In [24]:
import json
import time
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.utils.extmath import randomized_svd

In [25]:
INPUT_CSV  = "../dane/fra_cleaned_final.csv"
OUTPUT_CSV = "../dane/fra_cleaned_final_embeddings.csv"

NOTE_MAP = {}

CONFIG = dict(
    LAYER_WEIGHTS = {"Top": 1.0, "Middle": 1.0, "Base": 1.0}, # Równa waga dla każdej warstwy piramidy
    LENGTH_NORM_EXP = 0.0,                                    # Brak kary za długość listy składników
    MIN_DF_NOTE = 3,                                          # Nuta musi wystąpić min. 3 razy w całej bazie
    ACCORD_RANK_WEIGHTS = [1.0, 0.75, 0.56, 0.42, 0.32],      # Malejące wagi dla kolejnych głównych akordów, 0.75^k
    IDF_POWER = 0.5,                                          # Pierwiastek wygładzający wagę rzadkich nut, osłabia nuty dominujące np. piżmo
    A_ALPHA_ACCORD = 0.4,                                     # Metoda A: 40% wpływu akordów, 60% nut
    B_CTX_SMOOTH = 0.5,                                       # Metoda B: Wygładzanie korelacji między nutami, 1 - brak wygładzania, 0 - pełne wygładzenie
    B_ACCORD_GAIN = 1.5,                                      # Metoda B: Bonus dla zapachów z tej samej rodziny
    DIM = 256,                                                # Docelowa liczba wymiarów wektora zapachu (kompresja SVD)
    SEED = 42,                                                # Stałe ziarno losowości dla powtarzalności wyników
)

In [26]:
df = pd.read_csv(INPUT_CSV, sep=";", encoding="utf-8-sig")
N = len(df)  

print("kształt tabeli (wiersze, kolumny):", df.shape)
print("\nnazwy kolumn:")
print(df.columns.tolist())

print("\npierwsze trzy rekordy:")
print(df.loc[23577].T)

braki = df.isna().sum()
print("\nbraki danych:")
print(braki[braki > 0])

print("\npowtórzone adresy url:", df.url.duplicated().sum())
print("powtórzone pary (Perfume, Brand):", df.duplicated(["Perfume", "Brand"]).sum())

kształt tabeli (wiersze, kolumny): (24046, 17)

nazwy kolumn:
['url', 'Perfume', 'Brand', 'Country', 'Gender', 'Rating Value', 'Rating Count', 'Year', 'Top', 'Middle', 'Base', 'mainaccord1', 'mainaccord2', 'mainaccord3', 'mainaccord4', 'mainaccord5', 'Perfumers']

pierwsze trzy rekordy:
url             https://www.fragrantica.com/perfume/issey-miya...
Perfume                       L EAU D ISSEY POUR HOMME NOIR AMBRE
Brand                                                ISSEY MIYAKE
Country                                                     Japan
Gender                                                        men
Rating Value                                                  4.5
Rating Count                                                 1044
Year                                                       2016.0
Top                          saffron, cinnamon, nutmeg, coriander
Middle                           leather, cedar, papyrus, vetiver
Base                                   amber, vanill

In [27]:
def split_list(s):
    if not isinstance(s, str):
        return []
    return [x.strip().lower() for x in s.split(",")
            if x.strip() and x.strip().lower() != "unknown"]

for layer in ["Top", "Middle", "Base"]:
    df[layer + "_l"] = df[layer].map(split_list)

acc_cols = [f"mainaccord{i}" for i in range(1, 6)]
df["accords_l"] = df[acc_cols].apply(
    lambda row: [str(x).strip().lower() for x in row if isinstance(x, str) and x.strip()],
    axis=1,
)

NOTE_MAP = {}   # do zrobienia mapowanie nut szczegolowych na bardziej ogolne np. {"bourbon vanilla": "vanilla"}

for layer in ["Top", "Middle", "Base"]:
    df[layer + "_l"] = df[layer + "_l"].map(lambda lst: [NOTE_MAP.get(n, n) for n in lst])

all_notes = Counter(n for layer in ["Top_l", "Middle_l", "Base_l"]
                    for lst in df[layer] for n in lst)
all_acc = Counter(a for lst in df["accords_l"] for a in lst)
counts = np.array(list(all_notes.values()))

print(f"unikalne nuty: {len(all_notes)}")
print(f"  występujące tylko raz: {(counts == 1).sum()}")
print(f"  występujące co najmniej 3 razy: {(counts >= 3).sum()}")
print(f"  występujące co najmniej 50 razy: {(counts >= 50).sum()}")
print(f"unikalne akordy: {len(all_acc)}")

print("\nśrednia liczba nut w warstwie:",
      {l: round(float(df[l + "_l"].map(len).mean()), 2) for l in ["Top", "Middle", "Base"]})
print("\nile perfum ma ile akordów:")
print(df["accords_l"].map(len).value_counts().sort_index().to_string())
print("\nnajczęstsze nuty:", all_notes.most_common(10))

print("\nprzykład:")
print(df.loc[23577, ["Perfume", "Top", "Top_l", "accords_l"]].to_string())

unikalne nuty: 1664
  występujące tylko raz: 343
  występujące co najmniej 3 razy: 1135
  występujące co najmniej 50 razy: 366
unikalne akordy: 84

średnia liczba nut w warstwie: {'Top': 3.11, 'Middle': 3.3, 'Base': 3.5}

ile perfum ma ile akordów:
accords_l
1       13
2      101
3      274
4      591
5    23067

najczęstsze nuty: [('musk', 11013), ('bergamot', 8613), ('sandalwood', 8057), ('jasmine', 7741), ('amber', 7725), ('patchouli', 7250), ('vanilla', 6738), ('rose', 6249), ('cedar', 5625), ('mandarin orange', 4006)]

przykład:
Perfume               L EAU D ISSEY POUR HOMME NOIR AMBRE
Top                  saffron, cinnamon, nutmeg, coriander
Top_l              [saffron, cinnamon, nutmeg, coriander]
accords_l    [amber, leather, warm spicy, vanilla, woody]


In [28]:
# Każde perfumy zapisujemy jako rzadki wektor w przestrzeni CECH:
# jedna współrzędna na każdą nutę i jedna na każdy akord.
# Wartość współrzędnej = WAGA tej cechy w tych perfumach (0 = cechy brak).

# 1) cechy-nuty 
# Trzy decyzje projektowe (sprawdzimy je eksperymentem w komórce 6):
#   layer_w  - waga warstwy piramidy (Top / Middle / Base)
#   len_exp  - normalizacja długości: nuta dostaje wagę
#              waga_warstwy / (liczba nut w warstwie) ** len_exp
#              0 = bez karania długich piramid, 1 = pełne dzielenie przez długość
# Domyślnie: płaskie wagi 1/1/1 i len_exp = 0 (wartości wybrane w komórce 6).
def note_features(layer_w=None, len_exp=0.0):
    if layer_w is None:
        layer_w = {"Top": 1.0, "Middle": 1.0, "Base": 1.0}
    wynik = []
    for top, middle, base in zip(df.Top_l, df.Middle_l, df.Base_l):
        cechy = {}
        for lista, w in ((top, layer_w["Top"]),
                         (middle, layer_w["Middle"]),
                         (base, layer_w["Base"])):
            for nuta in lista:
                cechy[nuta] = cechy.get(nuta, 0.0) + w / (len(lista) ** len_exp)
        wynik.append(cechy)
    return wynik

# 2) cechy-akordy
# Akordy są uporządkowane od najbardziej wyczuwalnego, więc waga maleje z pozycją:
# [1.0, 0.75, 0.56, 0.42, 0.32] (zanik geometryczny 0.75^k).
def accord_features():
    wagi = CONFIG["ACCORD_RANK_WEIGHTS"]
    return [{akord: wagi[i] for i, akord in enumerate(lista[:5])}
            for lista in df.accords_l]

# 3) słowniki cech -> macierz rzadka 
def to_sparse(cechy_perfum, min_df=1):
    licznik = Counter(k for d in cechy_perfum for k in d)          # w ilu perfumach jest cecha
    vocab = sorted(k for k, v in licznik.items() if v >= min_df)   # zostają cechy z >= min_df perfum
    indeks = {k: i for i, k in enumerate(vocab)}

    wiersze, kolumny, wartosci = [], [], []
    for i, d in enumerate(cechy_perfum):
        for k, v in d.items():
            j = indeks.get(k)
            if j is not None:
                wiersze.append(i); kolumny.append(j); wartosci.append(v)

    X = sp.csr_matrix((wartosci, (wiersze, kolumny)),
                      shape=(len(cechy_perfum), len(vocab)))
    df_cechy = np.array([licznik[k] for k in vocab])   
    return X, vocab, df_cechy

#  4) obie tabele cech 
X_note, vocab_note, df_note = to_sparse(note_features(), CONFIG["MIN_DF_NOTE"])
X_acc,  vocab_acc,  df_acc  = to_sparse(accord_features())

print("tabela nut:    ", X_note.shape)
print("tabela akordów:", X_acc.shape)
print(f"średnio cech na perfumy: nuty {X_note.nnz / N:.1f}, akordy {X_acc.nnz / N:.1f}")
print(f"wypełnienie tabeli nut: {X_note.nnz / (X_note.shape[0] * X_note.shape[1]):.2%}")

# 5) podgląd 
i = 23577
def pokaz(X, vocab, i):
    wiersz = X[i]
    return {vocab[j]: round(float(v), 2) for j, v in zip(wiersz.indices, wiersz.data)}

print(f"\n{df.Perfume[i]}, nuty:  ", pokaz(X_note, vocab_note, i))
print(f"{df.Perfume[i]}, akordy:", pokaz(X_acc, vocab_acc, i))

# 6) kontrola: najpopularniejsze cechy 
print("\nnajczęstsze nuty (w ilu perfumach):",
      [(vocab_note[j], int(df_note[j])) for j in np.argsort(-df_note)[:5]])
print("najczęstsze akordy (w ilu perfumach):",
      [(vocab_acc[j], int(df_acc[j])) for j in np.argsort(-df_acc)[:5]])

tabela nut:     (24046, 1133)
tabela akordów: (24046, 84)
średnio cech na perfumy: nuty 9.8, akordy 4.9
wypełnienie tabeli nut: 0.87%

L EAU D ISSEY POUR HOMME NOIR AMBRE, nuty:   {'amber': 1.0, 'cedar': 1.0, 'cinnamon': 1.0, 'coriander': 1.0, 'leather': 1.0, 'nutmeg': 1.0, 'papyrus': 1.0, 'saffron': 1.0, 'tonka bean': 1.0, 'vanilla': 1.0, 'vetiver': 1.0}
L EAU D ISSEY POUR HOMME NOIR AMBRE, akordy: {'amber': 1.0, 'leather': 0.75, 'vanilla': 0.42, 'warm spicy': 0.56, 'woody': 0.32}

najczęstsze nuty (w ilu perfumach): [('musk', 10942), ('bergamot', 8608), ('sandalwood', 8007), ('amber', 7681), ('jasmine', 7669)]
najczęstsze akordy (w ilu perfumach): [('woody', 12523), ('citrus', 9483), ('aromatic', 7620), ('sweet', 7471), ('fruity', 6928)]


In [ ]:
# =====================================================================
# TEST ZASTĘPCZY "NUTY -> AKORDY" (AccAgree@10)
# =====================================================================
# Problem: nie mamy prawdziwych odpowiedzi, które perfumy są podobne.
# Pomysł: akordy opisują zapach ODBIERANY (głosy użytkowników), a nuty to skład
# DEKLAROWANY. Budujemy osadzenie wyłącznie z nut i sprawdzamy, czy sąsiedzi
# perfum mają podobne AKORDY do tych perfum, choć akordów nie użyliśmy.
#
#   AccAgree@10 = średnia po zapytaniach q z średniej po 10 sąsiadach n z cos(a_q, a_n)
#   (a = znormalizowany wektor rangowych wag akordów)
#
# Zastrzeżenie: akordy częściowo wynikają z nut, więc test jest częściowo "okrężny".
# To wciąż jedyny sygnał percepcyjny niezależny od naszych cech.

# 1. Przygotowanie "odpowiedzi" - gęsta macierz znormalizowanych akordów
Y_acc = normalize(X_acc).toarray()                    
rng = np.random.default_rng(CONFIG["SEED"])

# 2. Losowanie zapytań
Q = rng.choice(N, 1000, replace=False)  # 1000 losowych perfum-zapytań

# 3. Wyznaczenie poziomu losowego (dolna granica)
rand_baseline = float((Y_acc[Q] @ Y_acc.T).mean())    

def acc_agree(Z, k=10, queries=Q):
    """
    Funkcja oceniająca pokrycie akordów wśród k-najbliższych sąsiadów
    na podstawie macierzy osadzeń Z.
    """
    # Obliczenie kosinusów zapytań względem wszystkich perfum
    sim = Z[queries] @ Z.T                                            
    sim = sim.toarray() if sp.issparse(sim) else np.asarray(sim)
    
    # Wykluczenie perfum jako własnych sąsiadów
    sim[np.arange(len(queries)), queries] = -9        
    
    # Wybór k najbliższych sąsiadów (szybkie partycjonowanie zamiast sortowania)
    top = np.argpartition(-sim, k, axis=1)[:, :k]     
    
    # Obliczenie średniego podobieństwa akordów sąsiadów do zapytania
    wyniki = [(Y_acc[top[i]] @ Y_acc[q]).mean() for i, q in enumerate(queries)]
    return float(np.mean(wyniki))

# 4. Generowanie wektorów losowych do testu dolnej granicy
los = normalize(np.random.default_rng(0).normal(size=(N, 64)))

wyniki_testow = [
    {"Metryka": "Kształt macierzy Y_acc", "Wartość": str(Y_acc.shape)},
    {"Metryka": "Liczba zapytań (losowa próba)", "Wartość": len(Q)},
    {"Metryka": "Poziom losowy (dolna granica)", "Wartość": round(rand_baseline, 4)},
    {"Metryka": "Sam wzorzec (akordy jako wektory)", "Wartość": round(acc_agree(Y_acc), 4)},
    {"Metryka": "Losowe wektory", "Wartość": round(acc_agree(los), 4)} # powinny być bliskie dolnej granicy
]

df_wyniki = pd.DataFrame(wyniki_testow)
display(df_wyniki)

,Metryka,Wartość
0,Kształt macierzy Y_acc,"(24046, 84)"
1,Liczba zapytań (losowa próba),1000
2,Poziom losowy (dolna granica),0.2325
3,Sam wzorzec (akordy jako wektory),0.9187
4,Losowe wektory (powinno być bliskie dolnej gra...,0.2326


In [30]:
# =====================================================================
# ABLACJA: jak wybrać wagi nut bez etykiet?
# =====================================================================
# Trzy czynniki, każdy w trzech wariantach (3 x 3 x 3 = 27 wariantów):
#  1) waga warstwy piramidy: płaska 1/1/1, "baza ważniejsza", "głowa ważniejsza"
#     (intuicja: baza zostaje na skórze dłużej - czy dane to potwierdzają?)
#  2) normalizacja długości len_exp: 0 / 0.5 / 1
#     (czy nuta w krótkiej piramidzie ma ważyć więcej niż w długiej?)
#  3) moc IDF p: 0 / 0.5 / 1
#     IDF osłabia nuty pospolite (piżmo jest w ok. 46% perfum i mało mówi o
#     RÓŻNICY między perfumami):  idf(t) = ln((1+N)/(1+df(t))) + 1,  waga *= idf ** p
#     p = 0 oznacza brak osłabiania, p = 1 pełne osłabianie.
#
# Każdy wariant: tabela nut -> IDF -> normalizacja długości do 1 -> AccAgree@10.
# Cosinus liczymy na pełnych, rzadkich wektorach (bez kompresji SVD).

def idf_weights(df_cechy, power):
    return (np.log((1 + N) / (1 + df_cechy)) + 1.0) ** power

warianty_warstw = {
    "płaskie 1/1/1":              {"Top": 1.0, "Middle": 1.0, "Base": 1.0},
    "baza ważniejsza 0.5/1/1.2":  {"Top": 0.5, "Middle": 1.0, "Base": 1.2},
    "głowa ważniejsza 1.2/1/0.8": {"Top": 1.2, "Middle": 1.0, "Base": 0.8},
}

wiersze = []
t0 = time.time()
for nazwa, lw in warianty_warstw.items():
    for le in [0.0, 0.5, 1.0]:
        Xn, vn, dn = to_sparse(note_features(lw, le), CONFIG["MIN_DF_NOTE"])
        for p in [0.0, 0.5, 1.0]:
            S = normalize(Xn.multiply(idf_weights(dn, p)).tocsr())
            wiersze.append({"warstwy": nazwa, "len_exp": le, "idf_power": p,
                            "AccAgree10": round(acc_agree(S), 4)})
print(f"czas: {time.time() - t0:.0f} s")

abl = pd.DataFrame(wiersze).sort_values("AccAgree10", ascending=False).reset_index(drop=True)
print(f"\npoziom losowy: {rand_baseline:.4f}\n")

display(abl)

for czynnik in ["warstwy", "len_exp", "idf_power"]:
    print(f"\n wpływ czynnika: {czynnik}")
    tabela_grup = abl.groupby(czynnik)["AccAgree10"].agg(["mean", "max"]).round(4)
    display(tabela_grup)

czas: 11 s

poziom losowy: 0.2325



,warstwy,len_exp,idf_power,AccAgree10
0,płaskie 1/1/1,0.0,0.5,0.5361
1,głowa ważniejsza 1.2/1/0.8,0.0,0.5,0.5348
2,głowa ważniejsza 1.2/1/0.8,0.0,0.0,0.5278
3,płaskie 1/1/1,0.5,0.5,0.5247
4,płaskie 1/1/1,0.0,0.0,0.5220
5,głowa ważniejsza 1.2/1/0.8,0.5,0.5,0.5215
6,płaskie 1/1/1,0.0,1.0,0.5182
7,głowa ważniejsza 1.2/1/0.8,0.0,1.0,0.5165
8,głowa ważniejsza 1.2/1/0.8,0.5,0.0,0.5144
9,płaskie 1/1/1,0.5,0.0,0.5142



 wpływ czynnika: warstwy


,mean,max
warstwy,,
baza ważniejsza 0.5/1/1.2,0.4703,0.4856
głowa ważniejsza 1.2/1/0.8,0.5087,0.5348
płaskie 1/1/1,0.5097,0.5361



 wpływ czynnika: len_exp


,mean,max
len_exp,,
0.0,0.5098,0.5361
0.5,0.5006,0.5247
1.0,0.4782,0.4958



 wpływ czynnika: idf_power


,mean,max
idf_power,,
0.0,0.4946,0.5278
0.5,0.5046,0.5361
1.0,0.4894,0.5182


In [33]:
# =====================================================================
# METODA A: TF-IDF, łączenie bloków, kompresja SVD
# =====================================================================

#  1) budowa osadzenia 
def build_A(alpha, dim, idf_power, seed):
    # blok nut i blok akordów: IDF, potem skrócenie każdego wiersza do długości 1
    Sn = normalize(X_note.multiply(idf_weights(df_note, idf_power)).tocsr())
    Sa = normalize(X_acc.multiply(idf_weights(df_acc, idf_power)).tocsr())

    # sklejenie bloków; pierwiastki sprawiają, że wagi bloków to dokładnie alpha i 1-alpha
    V = sp.hstack([np.sqrt(alpha) * Sa, np.sqrt(1 - alpha) * Sn]).tocsr()

    # kompresja SVD do dim wymiarów, potem skrócenie wektorów do długości 1
    svd = TruncatedSVD(n_components=dim, random_state=seed)
    Z = normalize(svd.fit_transform(V))
    return Z, V, svd

t0 = time.time()
Z_A, V_A, svd_A = build_A(CONFIG["A_ALPHA_ACCORD"], CONFIG["DIM"],
                          CONFIG["IDF_POWER"], CONFIG["SEED"])

# Tworzenie Tabeli 1: Podsumowanie budowy
df_budowa = pd.DataFrame([
    {"Parametr": "Kształt przed kompresją (V_A)", "Wartość": str(V_A.shape)},
    {"Parametr": "Kształt po kompresji SVD (Z_A)", "Wartość": str(Z_A.shape)},
    {"Parametr": f"Wariancja wyjaśniona ({CONFIG['DIM']} osi)", "Wartość": f"{svd_A.explained_variance_ratio_.sum():.1%}"},
    {"Parametr": "Czas budowy [s]", "Wartość": round(time.time() - t0, 1)}
])

print(" 1. Statystyki budowy osadzenia ")
display(df_budowa)


#  2) kontrola: czy alpha naprawdę jest udziałem akordów? 
alpha = CONFIG["A_ALPHA_ACCORD"]
Sa = normalize(X_acc.multiply(idf_weights(df_acc, CONFIG["IDF_POWER"])).tocsr())
Sn = normalize(X_note.multiply(idf_weights(df_note, CONFIG["IDF_POWER"])).tocsr())
i, j = 1, 23577                                  # dowolne dwie perfumy
lewa  = (V_A[i] @ V_A[j].T).toarray()[0, 0]
prawa = alpha * (Sa[i] @ Sa[j].T).toarray()[0, 0] + (1 - alpha) * (Sn[i] @ Sn[j].T).toarray()[0, 0]
min_norm = np.linalg.norm(Z_A, axis=1).min().round(6)
max_norm = np.linalg.norm(Z_A, axis=1).max().round(6)

# Tworzenie Tabeli 2: Test matematyczny udziałów
df_kontrola = pd.DataFrame([
    {"Test": "Cos po sklejeniu bloków (lewa strona równania)", "Wynik": round(lewa, 4)},
    {"Test": "alpha*cos(akordy) + (1-alpha)*cos(nuty) (prawa)", "Wynik": round(prawa, 4)},
    {"Test": "Cos po kompresji SVD (ta sama para)", "Wynik": round(float(Z_A[i] @ Z_A[j]), 4)},
    {"Test": "Długości wektorów (min, max)", "Wynik": f"({min_norm}, {max_norm})"}
])

print("\n 2. Kontrola matematyczna udziału bloków ")
display(df_kontrola)


#  3) retencja: ile z 10 najbliższych sąsiadów przeżywa kompresję? 
#     Wzorzec: dokładny cosinus na wektorach sprzed kompresji (V_A).
samp = rng.choice(N, 500, replace=False)         # 500 losowych perfum
V_A_n = normalize(V_A)
dokladny = (V_A_n[samp] @ V_A_n.T).toarray()
przyblizony = Z_A[samp] @ Z_A.T
dokladny[np.arange(500), samp] = -9              # perfumy nie są własnym sąsiadem
przyblizony[np.arange(500), samp] = -9
top_dok = np.argpartition(-dokladny, 10, axis=1)[:, :10]
top_prz = np.argpartition(-przyblizony, 10, axis=1)[:, :10]
retencja = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(top_dok, top_prz)])

df_retencja = pd.DataFrame([
    {"Metryka": "Retencja top-10 względem dokładnego cosinusa", "Wynik": f"{retencja:.1%}"}
])

print("\n 3. Jakość kompresji")
display(df_retencja)


#  4) wpływ liczby wymiarów 
Sn_only = normalize(X_note.multiply(idf_weights(df_note, CONFIG["IDF_POWER"])).tocsr())

# Konstruujemy tabelę wyników dodając wiersze iteracyjnie do listy słowników
wyniki = [
    {"Wariant (wymiary)": "Poziom losowy (baseline)", "AccAgree@10 (tylko nuty)": round(rand_baseline, 4)},
    {"Wariant (wymiary)": "Bez kompresji (dokładny cosinus)", "AccAgree@10 (tylko nuty)": round(acc_agree(Sn_only), 4)}
]

for d in [32, 64, 128, 256]:
    Z = normalize(TruncatedSVD(d, random_state=CONFIG["SEED"]).fit_transform(Sn_only))
    wyniki.append({"Wariant (wymiary)": f"SVD {d} wymiarów", "AccAgree@10 (tylko nuty)": round(acc_agree(Z), 4)})

df_wymiary = pd.DataFrame(wyniki)

print("\n 4. Wpływ stopnia kompresji SVD na trafność")
display(df_wymiary)

 1. Statystyki budowy osadzenia 


,Parametr,Wartość
0,Kształt przed kompresją (V_A),"(24046, 1217)"
1,Kształt po kompresji SVD (Z_A),"(24046, 256)"
2,Wariancja wyjaśniona (256 osi),88.2%
3,Czas budowy [s],1.2



 2. Kontrola matematyczna udziału bloków 


,Test,Wynik
0,Cos po sklejeniu bloków (lewa strona równania),0.2964
1,alpha*cos(akordy) + (1-alpha)*cos(nuty) (prawa),0.2964
2,Cos po kompresji SVD (ta sama para),0.3394
3,"Długości wektorów (min, max)","(1.0, 1.0)"



 3. Jakość kompresji


,Metryka,Wynik
0,Retencja top-10 względem dokładnego cosinusa,66.7%



 4. Wpływ stopnia kompresji SVD na trafność


,Wariant (wymiary),AccAgree@10 (tylko nuty)
0,Poziom losowy (baseline),0.2325
1,Bez kompresji (dokładny cosinus),0.5361
2,SVD 32 wymiarów,0.4597
3,SVD 64 wymiarów,0.4906
4,SVD 128 wymiarów,0.5165
5,SVD 256 wymiarów,0.5303
